# Tiny Sparks — ACE-Step song test (FREE, Kaggle GPU)

Generates **The Colors Song** (chorus + 4 colour verses + chorus) with **real sung vocals** using the open-source
**ACE-Step** model (Apache-2.0), in 2 styles × 3 takes = **6 versions** to compare.

**Before you run (Kaggle):**
1. Right panel → **Settings → Accelerator → GPU T4 x2** (or P100).
2. Right panel → **Settings → Internet → ON** (needs a phone-verified Kaggle account).
3. **Run All**. First run downloads the model (~8 GB) — allow ~10 min, then ~1–3 min per take.
4. Listen to the players at the bottom, then download **`colors_song_takes.zip`** from the Output panel
   (or `/kaggle/working/`), and send me the best take's file name.

To try other songs: edit the **SETTINGS** cell (lyrics, styles, seeds, duration) and re-run from there.

In [ ]:
# 1) Install ACE-Step (takes a few minutes)
!python --version
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/ace-step/ACE-Step.git /kaggle/working/ACE-Step || echo "already cloned"
# Kaggle's Python is newer than ACE-Step's pins: unpin spacy (3.8.4 has no wheel for it) and keep
# Kaggle's own CUDA-matched torch/torchaudio/torchvision instead of reinstalling them.
!sed -i 's/^spacy==.*/spacy>=3.8.7/' /kaggle/working/ACE-Step/requirements.txt
!grep -v -E '^(torch|torchaudio|torchvision)$' /kaggle/working/ACE-Step/requirements.txt > /kaggle/working/req.txt
!pip install -q -r /kaggle/working/req.txt
!pip install -q --no-deps -e /kaggle/working/ACE-Step
import sys; sys.path.insert(0, "/kaggle/working/ACE-Step")


In [ ]:
# 2) SETTINGS — edit freely
LYRICS = """[chorus]
Colors, colors, all around!
Colors up and colors down!
Red and blue and yellow too,
I see colors, how about you?

[verse]
Red, red, red! Red like an apple!
Orange, orange! Orange like a carrot!
Yellow, yellow! Yellow like the sun!
Green, green, green! Green like a frog!

[chorus]
Colors, colors, all around!
Colors up and colors down!
Red and blue and yellow too,
I see colors, how about you?
"""

STYLES = {
    # name: the "tags" prompt (genre, instruments, voice, tempo)
    "nursery": "children's song, nursery rhyme, preschool, happy, bouncy, ukulele, xylophone, glockenspiel, "
               "hand claps, light drums, bright cheerful female vocal, clear diction, 118 bpm, major key",
    "kidspop": "kids pop, sing-along, energetic, catchy, kids choir, cheerful young female vocal, piano, "
               "acoustic guitar, hand claps, 124 bpm, major key, clean bright mix",
}
SEEDS = [11, 42, 2026]   # one take per seed, per style
DURATION = 75            # seconds (the lyrics above fit ~60-80 s)
STEPS = 60               # quality vs speed (27 = fast draft, 60 = good)
OUT = "/kaggle/working/out"

In [ ]:
# 3) Load the model (first run downloads the checkpoint)
import os, torch
from acestep.pipeline_ace_step import ACEStepPipeline
bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()   # T4/P100: False -> float32
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (enable GPU in Settings!)", "| bf16:", bf16)
pipe = ACEStepPipeline(
    checkpoint_dir=None,                 # auto-download to ~/.cache/ace-step/checkpoints
    dtype="bfloat16" if bf16 else "float32",
    torch_compile=False,
    cpu_offload=True,                    # keeps VRAM under ~8-10 GB (fits a 16 GB T4)
    overlapped_decode=True,
)

In [ ]:
# 4) Generate every style x seed
import glob, inspect, shutil, time
# T4 fix: cuDNN has no engine for the audio decoder's big convolutions ("GET was unable to find an
# engine"), so use PyTorch's native kernels. Costs only a few seconds per take.
torch.backends.cudnn.enabled = False
torch.cuda.empty_cache()
os.makedirs(OUT, exist_ok=True)
accepted = set(inspect.signature(pipe.__call__).parameters)
def generate(tags, seed, path):
    kw = dict(
        audio_duration=DURATION, prompt=tags, lyrics=LYRICS, infer_step=STEPS,
        guidance_scale=15, scheduler_type="euler", cfg_type="apg", omega_scale=10,
        manual_seeds=str(seed), guidance_interval=0.5, guidance_interval_decay=0.0,
        min_guidance_scale=3, use_erg_tag=True, use_erg_lyric=True, use_erg_diffusion=True,
        oss_steps="", guidance_scale_text=0.0, guidance_scale_lyric=0.0, save_path=path,
    )
    kw = {k: v for k, v in kw.items() if k in accepted}   # robust to small API changes
    before = set(glob.glob("/kaggle/working/**/*.wav", recursive=True) + glob.glob("./outputs/*.wav"))
    pipe(**kw)
    if not os.path.exists(path):          # some versions write their own file name: pick up the new file
        new = set(glob.glob("/kaggle/working/**/*.wav", recursive=True) + glob.glob("./outputs/*.wav")) - before
        if new: shutil.move(max(new, key=os.path.getmtime), path)

for name, tags in STYLES.items():
    for seed in SEEDS:
        path = f"{OUT}/colors-{name}-seed{seed}.wav"
        t = time.time()
        generate(tags, seed, path)
        print(f"{os.path.basename(path)}  {'OK' if os.path.exists(path) else 'MISSING'}  ({time.time()-t:.0f}s)")

In [ ]:
# 5) Listen
from IPython.display import Audio, display, Markdown
for p in sorted(glob.glob(f"{OUT}/*.wav")):
    display(Markdown(f"**{os.path.basename(p)}**")); display(Audio(p))

In [ ]:
# 6) Download: colors_song_takes.zip appears in the Output panel / /kaggle/working
shutil.make_archive("/kaggle/working/colors_song_takes", "zip", OUT)
print("-> /kaggle/working/colors_song_takes.zip")